# E1 - metric re-measurement sweep (self-contained, no GitHub token)

Implements `experiment 2/SPEC_E1_METRIC_REMEASUREMENT.md`. Generated by
`scripts/build_e1_notebook.py` from the current tree.

**What this does.** Re-measures Q on the three Stage-A adapters already in Drive
under a grid of alternative operationalizations (V1-V6). **No training, no
Stage B, no dependency on anyone else.** Estimated ~1-2 A100-hours.

**What it does not do.** It cannot test RQ1 - Delta-R is flat across these three
checkpoints however Q is measured. E1 is about whether *the detector* works.

## How to run
1. Runtime -> Change runtime type -> **A100 GPU** -> Save
2. Run cells 1-6 (setup + Drive restore). Mounting Drive asks for authorization.
3. Run cell 7 (pre-flight - fails fast on a provenance problem, costs seconds)
4. Run cell 8, then re-run cell 9 to follow the log.
5. Run cell 10 to copy results back to Drive.

## The gate
Step 1 re-runs the **unchanged reference arm** and must reproduce
`FINDING_Q_METRICS_7B_INSTRUCT.md` exactly (erank delta < 1e-4, dormant_frac
0.0 everywhere). Two independent ckpt-0 passes previously agreed bit for bit, so
any drift is a real environment change. If the gate fails the driver **stops** -
do not accept and note it.

## Why the source is embedded
The PAT that `colab/02_transfer_T_and_qmetrics.ipynb` clones with is confirmed
broken (HTTP 403, `Write access to repository not granted`). This notebook
carries the source it needs as a blob instead, same as
`00_phase0_selfcontained.ipynb`. **The blob is a SNAPSHOT** - regenerate with
`scripts/build_e1_notebook.py` if `experiment 2/src/`, the driver, the instruct
config, or `eaaj-pilot/src/*.py` changes.

Model: `Qwen/Qwen2.5-7B-Instruct`, LoRA r=16 alpha=32, base bf16 / adapter fp32.


In [ ]:
#@title 1 GPU gate - refuse to continue on unsuitable hardware
# Deliberately does NOT import torch. torch imports numpy, and this notebook
# later installs a pinned numpy that is a major version ahead of Colab's; if
# numpy is already resident when that install lands, every downstream import
# dies ("cannot import name '_center' from 'numpy._core.umath'") and the only
# cure is a kernel restart, which turns Run all into a two-pass process that
# needs a human to press it again. On 2026-08-16 that cost two runtimes to idle
# reclamation. Querying nvidia-smi in a subprocess keeps this cell's fail-fast
# value without touching the numpy that cell 3 is about to replace.
import subprocess, sys

_q = subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,memory.total,compute_cap",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True)
if _q.returncode != 0 or not _q.stdout.strip():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 GPU -> Save")

_name, _mem_mib, _cap = [f.strip() for f in _q.stdout.strip().splitlines()[0].split(",")]
total_gb = int(_mem_mib) / 1024
cap_major = int(float(_cap))
print(f"GPU          : {_name}")
print(f"compute cap  : {_cap}")
print(f"total memory : {total_gb:.1f} GiB")

problems = []
if cap_major < 8:
    problems.append(
        f"Architecture too old (cap {_cap}). The recipe uses bfloat16, which "
        f"needs Ampere (8.0) or newer. T4/V100 will not work.")
if total_gb < 35:
    problems.append(
        f"Only {total_gb:.1f} GiB of VRAM. Qwen2.5-7B in bf16 is ~15 GiB of weights "
        f"before any group-8 generation state. Measured evidence from 2026-08-16: "
        f"the 0.5B track at this same group-8 geometry already needed ~27 GiB and "
        f"OOM'd on a 22 GiB L4. Use A100.")

if problems:
    print("\nUnsuitable GPU:")
    for p in problems:
        print("  -", p)
    raise SystemExit("GPU gate failed")
print("\nGPU gate passed (bf16 support is re-confirmed against torch in cell 4)")


In [ ]:
#@title 2 Unpack embedded source (no GitHub token required)
# This notebook carries the repo files it needs as a gzip+base64 blob instead of
# cloning the private repo. Same trick the v9 4070 probe used on 2026-08-16, for
# the same reason: the private-repo clone needs a PAT, and getting that PAT's
# fine-grained permissions right repeatedly failed (and leaked the token into
# cell output twice before it was sanitized). No token means neither failure mode
# can happen. The layout below reproduces the sibling-directory structure
# src/pipeline.py depends on (EXP2_ROOT/.. must contain eaaj-pilot/src).
import base64, gzip, io, os, sys, json, tarfile
from pathlib import Path

_B64 = """
...
"""

REPO_DIR = "/content/RLVR"
os.makedirs(REPO_DIR, exist_ok=True)
_raw = gzip.decompress(base64.b64decode("".join(_B64.split())))
with tarfile.open(fileobj=io.BytesIO(_raw), mode="r") as _tar:
    _tar.extractall(REPO_DIR)

EXP2_DIR = f"{REPO_DIR}/experiment 2"
os.makedirs(f"{EXP2_DIR}/data", exist_ok=True)
for _p in sorted(Path(REPO_DIR).rglob("*")):
    if _p.is_file():
        print(" ", _p.relative_to(REPO_DIR))
print("\nunpacked into", REPO_DIR)


In [ ]:
#@title 3 Install pinned dependencies, keeping the resident numpy (2-4 min)
# DEVIATION LOGGED 2026-08-16: requirements.txt pins numpy==2.3.5 to mirror the
# local env, but Colab's kernel has numpy RESIDENT from interpreter startup
# (2.0.2 today) - before any user cell runs. Upgrading numpy across versions
# under a live kernel breaks every later import ("cannot import name '_center'
# from 'numpy._core.umath'"), and the only cure is a kernel restart, which makes
# Run all two-pass and cost three runtimes to idle reclamation on 2026-08-16.
# So numpy is deliberately HELD at the resident version via a pip constraint.
# The scientific manifest (trl/transformers/datasets/accelerate/peft/
# bitsandbytes/pylatexenc, checked in cell 4) is installed exactly as pinned;
# numpy is not part of that contract and its exact patch version does not enter
# GRPO semantics. torch is NOT reinstalled - Colab's CUDA build is kept.
import importlib.metadata as _md

_resident_numpy = _md.version("numpy")
_req = open("/content/RLVR/experiment 2/requirements.txt").read().splitlines()
_kept = [l for l in _req if not l.strip().lower().startswith("numpy")]
open("/tmp/req_no_numpy.txt", "w").write("\n".join(_kept))
open("/tmp/constraints.txt", "w").write(f"numpy=={_resident_numpy}\n")
print(f"holding numpy at resident {_resident_numpy}; installing the rest as pinned")

%pip install -q -r /tmp/req_no_numpy.txt -c /tmp/constraints.txt
print("\nInstall finished.")


In [ ]:
#@title 3b Assert the kernel is clean (must be a no-op)
# With the cell-1 reorder nothing imports numpy before cell 3 installs it, so
# the loaded and installed versions must agree on the first pass and Run all
# completes without a restart. If this ever fires, the ordering invariant broke:
# something above imported numpy (directly, or via torch/pandas) before the
# install. Fix the ordering rather than adding a restart - a restart makes Run
# all two-pass, which needs an operator present and cost two runtimes to idle
# reclamation on 2026-08-16.
import importlib.metadata as _md
import numpy as _np

_installed = _md.version("numpy")
if _np.__version__ != _installed:
    raise SystemExit(
        f"ORDERING BROKEN: numpy {_np.__version__} was already resident before "
        f"the install put {_installed} on disk. Something above cell 3 imports "
        "numpy. Restarting would work but re-introduces the two-pass problem - "
        "fix the import order instead.")
print(f"kernel clean: numpy loaded == installed == {_installed}, single pass OK")


In [ ]:
#@title 4 Environment check - versions must match the pinned manifest
import importlib.metadata as md
import sys
import torch

EXPECTED = {"trl": "1.6.0", "transformers": "5.13.0", "datasets": "5.0.0",
            "accelerate": "1.14.0", "peft": "0.15.2", "bitsandbytes": "0.49.2",
            "pylatexenc": "2.10"}

print("python", ".".join(map(str, sys.version_info[:3])))
print("torch ", torch.__version__, f"(CUDA {torch.version.cuda})  <- Colab preinstalled")
print()
mismatched = []
for pkg, want in EXPECTED.items():
    try:
        got = md.version(pkg)
    except Exception:
        got = "MISSING"
    if got != want:
        mismatched.append(pkg)
    print(f"  {'ok ' if got == want else 'BAD'} {pkg:<14} want {want:<10} got {got}")

if mismatched:
    print("\nVersion mismatch:", ", ".join(mismatched))
    print("TRL version drift can silently change GRPO semantics. Report before trusting results.")
else:
    print("\nAll pinned packages match the manifest.")
import numpy as _np_v
print(f"numpy held at resident {_np_v.__version__} (deviation logged in cell 3; not part of the pinned manifest)")

# Import the bundled modules now, in-process, so a missing dependency surfaces in
# seconds rather than after the 7B download.
sys.path.insert(0, "/content/RLVR/experiment 2")
import src.guru_data, src.guru_reward, src.pipeline  # noqa: F401
from vendor.reasoning360_reward_score import codeio, naive_dapo  # noqa: F401
print("Import smoke test passed: bundled data/reward/pipeline modules load cleanly.")


In [ ]:
#@title 5 Pre-download Qwen/Qwen2.5-7B-Instruct (~15 GB - several minutes)
# E1 measures the INSTRUCT checkpoints, so this downloads the Instruct model,
# not the base model notebook 00 pulls. Isolating the download from the
# measurement gives one clean progress bar instead of a stall mid-sweep.
import subprocess, sys, time

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
print("model:", MODEL_ID)
code = (
    "from transformers import AutoTokenizer, AutoConfig\n"
    "from huggingface_hub import snapshot_download\n"
    f"AutoTokenizer.from_pretrained({MODEL_ID!r})\n"
    f"AutoConfig.from_pretrained({MODEL_ID!r})\n"
    f"snapshot_download({MODEL_ID!r})\n")

for attempt in range(1, 6):
    print(f"Attempt {attempt}/5 ...", flush=True)
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if r.returncode == 0:
        print("Download successful.")
        break
    print("Failed:", (r.stderr or "").strip().splitlines()[-1:] or "(no stderr)")
    if attempt < 5:
        time.sleep(15)
else:
    raise SystemExit("Could not download from Hugging Face after 5 attempts.")


In [ ]:
#@title 6 Restore config, splits and the three Stage-A adapters from Drive
# Mount IN-PROCESS first. The restore driver runs as a subprocess, and
# google.colab.drive.mount() cannot complete its interactive authorization from
# a subprocess - it talks to the notebook frontend over the kernel's comms
# channel, which a child process does not have. Mounting here means the driver
# finds /content/drive/MyDrive already present and skips its own mount call.
# This popup is the one manual step in the notebook.
from google.colab import drive
drive.mount("/content/drive")

# Verbatim driver, hash-verified: a stale or corrupted Drive copy fails loudly
# rather than silently changing the experiment.
import subprocess, sys
r = subprocess.run([sys.executable, "/content/RLVR/experiment 2/drivers/00_restore_from_drive.py"],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit("restore failed")


In [ ]:
#@title 7 Pre-flight: provenance + probe, no GPU work
# Fails fast on a bad config hash or split hash BEFORE the model is loaded,
# so a provenance problem costs seconds instead of an A100-hour.
import sys
sys.path.insert(0, "/content/RLVR/experiment 2")
sys.argv = ["04_e1_metric_sweep.py"]
import importlib.util
spec = importlib.util.spec_from_file_location(
    "e1driver", "/content/RLVR/experiment 2/drivers/04_e1_metric_sweep.py")
e1 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(e1)

config, splits = e1.load_provenance()
prompts = e1.probe_prompts(config, splits)
print("probe ready:", len(prompts), "prompts")
print("layers:", config["measurement"]["layers"],
      "| batch:", config["measurement"]["batch_size"])


In [ ]:
#@title 8 Run E1 steps 1-3 (reference-arm gate, then the sweep)
# Step 1 is the GATE: if the reference arm does not reproduce
# FINDING_Q_METRICS_7B_INSTRUCT.md exactly, this raises and nothing downstream
# runs (spec section 6 gate 1). Steps 2-3 involve no generation.
#
# Detached so an idle-reclaim of the browser tab does not kill the run; the log
# is tailed in the next cell.
import subprocess, sys
LOG = "/content/e1_sweep.log"
proc = subprocess.Popen(
    [sys.executable, "/content/RLVR/experiment 2/drivers/04_e1_metric_sweep.py"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("launched pid", proc.pid, "->", LOG)


In [ ]:
#@title 9 Follow the run
!tail -n 60 /content/e1_sweep.log


In [ ]:
#@title 10 Copy E1 outputs back to Drive
# The sweep's JSONs, per-unit score vectors and summary.csv are small. Copying
# them to Drive means a VM recycle does not cost the A100-hour that produced them.
import shutil
from pathlib import Path
RUN = "exp2_colab_guru_math7b_instruct_group8_e33527592dd9"
src = Path("/content/outputs") / RUN / "measurements" / "e1_sweep"
dst = Path("/content/drive/MyDrive/eaaj-exp2-checkpoints/e1_sweep")
if not src.is_dir():
    raise SystemExit(f"nothing at {src} yet - has the sweep finished?")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("copied to", dst)
print(sorted(p.name for p in dst.iterdir()))
